<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v3/blob/main/notebooks/admin/93_PROCESS_CLOUDFLARE_JOBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantvesting | Process Cloudflare Customer Jobs

Admin-only bridge for the current beta. Customer uploads are stored by Cloudflare/R2; this notebook downloads pending jobs, runs the existing Quantvesting engine in Colab, and publishes a customer-safe result back to Cloudflare.

> The customer never needs to open this notebook.

In [1]:
!pip install ta -qq

  Preparing metadata (setup.py) ... done


In [2]:
from pathlib import Path
import json
import os
import shutil
import tempfile
import urllib.request
import urllib.error
from getpass import getpass
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/My Drive/quantvesting_v3'
os.chdir(project_path)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src' / 'quantvesting').exists():
    raise RuntimeError('Run this notebook from the Quantvesting repository root in Colab.')

os.environ['PYTHONPATH'] = str(REPO_ROOT / 'src')
import sys
sys.path.insert(0, str(REPO_ROOT / 'src'))

WORKER_URL = input('Cloudflare Worker URL: ').strip().rstrip('/')
ADMIN_TOKEN = getpass('Cloudflare ADMIN_TOKEN: ').strip()
if not WORKER_URL or not ADMIN_TOKEN:
    raise ValueError('Worker URL and admin token are required.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloudflare Worker URL: https://quantvesting-v3.rathi-ankit.workers.dev/
Cloudflare ADMIN_TOKEN: ··········


In [3]:
def request_json(path, method='GET', payload=None):
    url = WORKER_URL + path
    data = None if payload is None else json.dumps(payload).encode('utf-8')

    headers = {
        'X-Admin-Token': ADMIN_TOKEN,
        'Accept': 'application/json',
        'User-Agent': 'Quantvesting-Colab/1.1',
    }
    if data is not None:
        headers['Content-Type'] = 'application/json'

    req = urllib.request.Request(
        url=url,
        data=data,
        headers=headers,
        method=method,
    )

    with urllib.request.urlopen(req, timeout=60) as response:
        body = response.read().decode('utf-8')
        return json.loads(body)


def request_bytes(path):
    url = WORKER_URL + path
    headers = {
        'X-Admin-Token': ADMIN_TOKEN,
        'Accept': '*/*',
        'User-Agent': 'Quantvesting-Colab/1.1',
    }
    req = urllib.request.Request(
        url=url,
        headers=headers,
        method='GET',
    )

    with urllib.request.urlopen(req, timeout=60) as response:
        return response.read()

print(request_json('/api/health'))


{'service': 'quantvesting', 'status': 'ok', 'mode': 'customer-gateway'}


In [4]:
from quantvesting import Quantvesting, load_config, load_market_data, load_portfolio_data
from quantvesting.dashboard import build_terminal_model
from quantvesting.reporting import summary_to_dict
from quantvesting.decisions import add_portfolio_actions, add_prospect_actions, capital_rotation_actions
from quantvesting.prospects import run_prospect_analysis
from quantvesting.portfolio import run_portfolio_analysis

CONFIG = load_config(REPO_ROOT / 'config' / 'strategy.yaml')
MARKET_DIR = REPO_ROOT / 'market_data'


In [5]:
def clean_json(value):
    import numpy as np
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, pd.DataFrame):
        return clean_json(value.replace({float('nan'): None}).to_dict(orient='records'))
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and (pd.isna(value) or value in (float('inf'), float('-inf'))):
        return None
    return value

def process_job(job):
    job_id = job['job_id']
    portfolio_id = job['portfolio_id']
    root = Path(tempfile.mkdtemp(prefix=f'qv_{portfolio_id}_'))
    portfolio_dir = root / 'portfolio_data' / portfolio_id
    portfolio_dir.mkdir(parents=True, exist_ok=True)
    try:
        csv_bytes = request_bytes(f'/api/admin/jobs/{job_id}/input')
        input_path = root / 'upload.csv'
        input_path.write_bytes(csv_bytes)

        qv = Quantvesting(CONFIG)
        qv.onboard_portfolio_csv(input_path, portfolio_dir)

        market_data = load_market_data(MARKET_DIR)
        portfolio_data = load_portfolio_data(portfolio_dir, portfolio_id=portfolio_id)
        prospects = qv.prospects(market_data, portfolio_data=portfolio_data, portfolio_id=portfolio_id)
        portfolio, summary = qv.portfolio(market_data, portfolio_data=portfolio_data, portfolio_id=portfolio_id)
        portfolio_actions = qv.portfolio_actions(portfolio)
        prospect_actions = qv.prospect_actions(prospects)
        rotation = qv.capital_rotation(prospects, portfolio)
        terminal = build_terminal_model(portfolio, prospects, summary, rotation, portfolio_data)

        # Keep the web payload intentionally compact. Full DataFrames remain
        # available through the notebook workflow and can be exposed later.
        payload = {
            'status': 'READY',
            'portfolio_id': portfolio_id,
            'run_id': summary.get('run_id'),
            'engine_version': summary.get('engine_version'),
            'strategy_version': summary.get('strategy_version'),
            'terminal': clean_json({
                'current_value': terminal.get('current_value'),
                'deployed_value': terminal.get('deployed_value'),
                'cagr_xirr': terminal.get('xirr'),
                'target_value': terminal.get('target_value'),
                'target_profit': terminal.get('target_profit'),
                'target_profit_pct': terminal.get('target_profit_pct'),
                'in_portfolio_health_pct': terminal.get('portfolio_health_pct'),
                'core_allocation_pct': terminal.get('core_allocation_pct'),
                'legacy_allocation_pct': terminal.get('legacy_allocation_pct'),
                'out_of_universe_allocation_pct': terminal.get('out_of_universe_allocation_pct'),
                'top5_concentration_pct': terminal.get('top5_concentration_pct'),
                'top10_concentration_pct': terminal.get('top10_concentration_pct'),
                'top20_concentration_pct': terminal.get('top20_concentration_pct'),
            }),
            'data_quality': clean_json(summary.get('validation', {})),
        }
        request_json(f'/api/admin/jobs/{job_id}/result', method='POST', payload=payload)
        return payload
    except Exception as exc:
        request_json(f'/api/admin/jobs/{job_id}/failure', method='POST', payload={'error': str(exc)})
        raise
    finally:
        shutil.rmtree(root, ignore_errors=True)

jobs = request_json('/api/admin/jobs').get('jobs', [])
print(f'Pending jobs: {len(jobs)}')
for job in jobs:
    print(f"Processing {job['job_id']} / {job['portfolio_id']}")
    result = process_job(job)
    print('READY:', result['portfolio_id'])


Pending jobs: 1
Processing 1c7ba5a8-87d5-4124-b96c-8be71932eee5 / customer-1c7ba5a8
READY: customer-1c7ba5a8
